# 5. Pace, momentum, and outcomes


### Purpose

Investigate whether recent puck-movement pace signals subsequent offensive pressure, and whether pace adds information beyond recent shot attempts and game context.

Treat pace, momentum, and outcomes as distinct concepts. Pace describes the speed of observed puck movement; momentum is a working hypothesis about sustained attacking advantage, not an established consequence of playing faster. Assess future shot attempts and dangerous-location attempts as primary outcomes, with goals as a secondary outcome.

### Expected outcome

A reproducible calculation of recent pace and attacking-pressure indicators, an assessment of their relationship with subsequent offense, and game timelines that help an analyst inspect the underlying sequences. Report sample coverage and uncertainty, including results where pace provides little additional information.


## 5.1. Setup and load processed data

Load the augmented event context and reference tables from notebooks 01-02, plus transition and sequence pace from notebook 03. Read the pace export manifest to retain the timing assumptions, units, and aggregation choices.

Recorded clocks remain the source evidence. Modeled within-second times are estimates; inferred possession spans are not measured continuous puck-control durations. No momentum calculation is performed in this setup section.


In [32]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)

ROOT_DIR = Path("..")
PROCESSED_DATA_DIR = ROOT_DIR / "data" / "processed"

pace_manifest = json.loads(
    (PROCESSED_DATA_DIR / "pace_export_manifest.json").read_text(encoding="utf-8")
)
reference_tables = {
    name: pd.read_parquet(PROCESSED_DATA_DIR / f"{name}.parquet")
    for name in ["teams", "players", "games", "event_types", "events_augmented"]
}
pace_tables = {
    name: pd.read_parquet(PROCESSED_DATA_DIR / f"{name}.parquet")
    for name in ["pace_transitions", "pace_sequences"]
}

display(pd.DataFrame([
    {"table": name, "rows": len(table), "columns": len(table.columns)}
    for name, table in {**reference_tables, **pace_tables}.items()
]))
print("Timing assumptions:", pace_manifest["timing"])


,table,rows,columns
0,teams,20,3
1,players,435,4
2,games,34,5
3,event_types,13,2
4,events_augmented,86670,27
5,pace_transitions,63298,25
6,pace_sequences,16046,22


Timing assumptions: Recorded clocks preserved; speeds use notebook 02 modeled clocks. Fractions are estimated, not measured.


## 5.2. Define the game timeline

Select a game and use all manpower situations. Use `SAMPLE_SECONDS` for sampling and `WINDOW_SECONDS` for the trailing window. Windows reset each period and use available history at period starts.

Regulation periods use 20-minute clocks. For overtime, use the maximum recorded clock as the observed period length; it may underestimate the scheduled length. Stop each period at its last observed event, excluding intermissions. Modeled timestamps determine event placement and transition completion.


In [39]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

GAME_ID = int(reference_tables["games"]["game_id"].iloc[1])
WINDOW_SECONDS = 120
SAMPLE_SECONDS = 5

assert WINDOW_SECONDS > 0 and SAMPLE_SECONDS > 0

game = reference_tables["games"].set_index("game_id").loc[GAME_ID]
team_names = reference_tables["teams"].set_index("team_id")["team_name"]
home_name = team_names.loc[game.home_team_id]
away_name = team_names.loc[game.away_team_id]

game_events = reference_tables["events_augmented"].loc[
    reference_tables["events_augmented"]["game_id"].eq(GAME_ID)
].sort_values("event_id").copy()
assert not game_events.empty, "Select a game present in the event table."

period_lengths = game_events.groupby("period")["clock_seconds"].max().sort_index()
period_lengths.loc[period_lengths.index <= 3] = 1200
period_ends = period_lengths - game_events.groupby("period")["modeled_clock_seconds"].min()
period_offsets = period_ends.cumsum().shift(fill_value=0)

game_events["period_elapsed_seconds"] = (
    game_events["period"].map(period_lengths) - game_events["modeled_clock_seconds"]
)
game_events["game_elapsed_seconds"] = (
    game_events["period"].map(period_offsets) + game_events["period_elapsed_seconds"]
)

transitions = pace_tables["pace_transitions"].loc[
    pace_tables["pace_transitions"]["game_id"].eq(GAME_ID)
    & pace_tables["pace_transitions"]["exclusion_reason"].isna()
].copy()
transitions["end_elapsed_seconds"] = (
    transitions["period"].map(period_lengths) - transitions["end_modeled_clock_seconds"]
)
# Attach the manpower context from each completed transition's end event.
transitions = transitions.merge(
    game_events[["event_id", "team_skaters", "opponent_skaters"]],
    left_on="end_event_id", right_on="event_id", validate="many_to_one",
)
transitions["situation"] = (
    transitions["team_skaters"].astype(str) + "v" + transitions["opponent_skaters"].astype(str)
)
print(f"{home_name} vs {away_name} | {game.game_date} | all situations")


Metropolitan Riveters vs Toronto Six | 2021-01-23 00:00:00 | all situations


## 5.3. Calculate rolling possession pace

For each team, pool distance and modeled duration from eligible transitions ending in `(sample - window, sample]` within the current period. Include full completed transitions, including same-second movement. No future endpoints, minimum-exposure threshold, interpolation, or carried values are used.

Store team pace as zero whenever pooled modeled duration is not positive, including empty windows and zero-duration movements. Home-minus-away pace is calculated directly from the stored team values. Transition counts distinguish empty-window zeros from measured zero pace. Hover exposes duration, transition count, and manpower composition; a small denominator can produce an unstable estimate. This is pace advantage, not yet a validated measure of momentum.


In [40]:
pace_rows = []
for period, end_time in period_ends.items():
    period_transitions = transitions.loc[transitions["period"].eq(period)]
    samples = np.unique(np.append(np.arange(0, end_time, SAMPLE_SECONDS), end_time))
    for sample in samples:
        window = period_transitions.loc[
            period_transitions["end_elapsed_seconds"].gt(sample - WINDOW_SECONDS)
            & period_transitions["end_elapsed_seconds"].le(sample)
        ]
        row = {
            "period": period,
            "period_elapsed_seconds": sample,
            "game_elapsed_seconds": period_offsets.loc[period] + sample,
            "clock_seconds_remaining": period_lengths.loc[period] - sample,
            "available_window_seconds": min(sample, WINDOW_SECONDS),
        }
        for side, team_id in [("home", game.home_team_id), ("away", game.away_team_id)]:
            movements = window.loc[window["possession_team_id"].eq(team_id)]
            distance = float(movements["distance_ft"].sum())
            duration = float(movements["modeled_elapsed_seconds"].sum())
            row[f"{side}_distance_ft"] = distance
            row[f"{side}_movement_seconds"] = duration
            row[f"{side}_transitions"] = len(movements)
            row[f"{side}_pace_ft_s"] = (
                distance / duration if duration > 0 else 0.0
            )
            row[f"{side}_situations"] = ", ".join(
                f"{situation}: {count}"
                for situation, count in movements["situation"].value_counts().items()
            ) or "No movements"
        row["pace_advantage_ft_s"] = row["home_pace_ft_s"] - row["away_pace_ft_s"]
        pace_rows.append(row)

pace_timeline = pd.DataFrame(pace_rows)
display(pace_timeline.groupby("period").agg(
    samples=("game_elapsed_seconds", "size"),
    home_estimates=("home_pace_ft_s", "count"),
    away_estimates=("away_pace_ft_s", "count"),
    comparisons=("pace_advantage_ft_s", "count"),
))


,samples,home_estimates,away_estimates,comparisons
period,,,,
1,241,241,241,241
2,241,241,241,241
3,241,241,241,241


## 5.4. Plot pace and pace advantage

Show each team's rolling total pace and the home-minus-away advantage on a shared game-clock axis. Draw periods separately so no line connects across a window reset. Annotate all recorded goals and home/away manpower changes. Empty team windows use the stored zero pace in both plots. Zero-duration windows also use stored zero pace. Hover identifies empty windows by transition count.


In [41]:
HOME_COLOR = "#2563eb"
AWAY_COLOR = "#ea580c"
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.07,
    row_heights=[0.6, 0.4],
    subplot_titles=["Rolling possession pace", "Home minus away pace"],
)

for period, samples in pace_timeline.groupby("period", sort=False):
    for side, name, color in [("home", home_name, HOME_COLOR), ("away", away_name, AWAY_COLOR)]:
        fig.add_trace(go.Scatter(
            x=samples["game_elapsed_seconds"] / 60,
            y=samples[f"{side}_pace_ft_s"], mode="lines", name=name,
            legendgroup=side, showlegend=bool(period == pace_timeline["period"].iloc[0]),
            line=dict(color=color), connectgaps=False,
            customdata=samples[[f"{side}_movement_seconds", f"{side}_transitions",
                                f"{side}_situations", "available_window_seconds"]].to_numpy(),
            hovertemplate=(
                "%{x:.2f} game min<br>Pace: %{y:.2f} ft/s"
                "<br>Movement time: %{customdata[0]:.2f} s"
                "<br>Transitions: %{customdata[1]}<br>Situations: %{customdata[2]}"
                "<br>Available window: %{customdata[3]:.1f} s<extra>%{fullData.name}</extra>"
            ),
        ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=samples["game_elapsed_seconds"] / 60, y=samples["pace_advantage_ft_s"],
        mode="lines", line=dict(color="#475569"), name="Pace advantage",
        showlegend=False, connectgaps=False,
        customdata=np.column_stack([
            np.where(samples["home_transitions"].eq(0), "No movements (zero)",
                     np.where(samples["home_movement_seconds"].le(0), "Zero duration (zero)", "Observed")),
            np.where(samples["away_transitions"].eq(0), "No movements (zero)",
                     np.where(samples["away_movement_seconds"].le(0), "Zero duration (zero)", "Observed")),
        ]),
        hovertemplate=("%{x:.2f} game min<br>Home - away: %{y:.2f} ft/s"
                       "<br>Home: %{customdata[0]}<br>Away: %{customdata[1]}<extra></extra>"),
    ), row=2, col=1)

# Align every row and mark each period's observed start.
for period, offset in period_offsets.items():
    fig.add_vline(x=float(offset) / 60, line_dash="dash", line_color="#94a3b8")
    fig.add_annotation(x=float(offset) / 60, y=1.02, xref="x", yref="paper",
                       text=f"P{period}", showarrow=False, xanchor="left")
fig.add_hline(y=0, line_color="#94a3b8", row=2, col=1)

# Home/away counts allow consistent annotation when the credited team changes.
game_events["home_skaters"] = game_events["team_skaters"].where(
    game_events["is_home"], game_events["opponent_skaters"]
)
game_events["away_skaters"] = game_events["opponent_skaters"].where(
    game_events["is_home"], game_events["team_skaters"]
)
previous_counts = game_events.groupby("period")[["home_skaters", "away_skaters"]].shift()
changes = game_events[["home_skaters", "away_skaters"]].ne(previous_counts).any(axis=1)
for event in game_events.loc[changes].itertuples():
    fig.add_vline(x=event.game_elapsed_seconds / 60, row=2, col=1,
                  line_color="#64748b", line_dash="dot")
    fig.add_annotation(x=event.game_elapsed_seconds / 60, y=0, row=2, col=1,
                       text=f"{event.home_skaters}v{event.away_skaters}",
                       showarrow=False, yshift=22, font=dict(size=9))

for goal in game_events.loc[game_events["event"].isin(["Goal", "Penalty Shot Goal"])].itertuples():
    color = HOME_COLOR if goal.team_id == game.home_team_id else AWAY_COLOR
    fig.add_vline(x=goal.game_elapsed_seconds / 60, line_color=color, line_dash="dot")
    fig.add_annotation(x=goal.game_elapsed_seconds / 60, y=0, row=2, col=1,
                       text=f"Goal: {team_names.loc[goal.team_id]}", showarrow=True,
                       arrowcolor=color, font=dict(color=color))

fig.update_layout(
    title=f"{home_name} vs {away_name} | trailing {WINDOW_SECONDS}s | all situations",
    template="plotly_white", height=650, hovermode="closest",
    legend=dict(orientation="h", y=1.13), margin=dict(t=130),
)
fig.update_yaxes(title_text="Pace (ft/s)", row=1, col=1)
fig.update_yaxes(title_text="Advantage (ft/s)", row=2, col=1)
fig.update_xaxes(range=[0, float(period_ends.sum()) / 60])
fig.update_xaxes(title_text="Game-clock elapsed minutes (intermissions excluded)", row=2, col=1)
fig.show()
